# Airport-OCR — full VOBL pipeline (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yashpatle23/Airport-OCR/blob/feat/airport-ocr-poc/notebooks/Airport_OCR_Full_Pipeline.ipynb)

**Problem scope.** Convert the unstructured BLR (VOBL) Aerodrome Chart PDF into
structured, machine-readable data:

`PDF -> Extract -> Identify -> Structure -> Search`

This one notebook runs the whole thing:

- **Step 1 — Copy file:** download `VOBL-ADC.pdf` (with an upload fallback).
- **Step 2 — Extract only:** (1) airport, (2) runways, (3) taxiways,
  (4) runway holding positions, (5) airport coordinates/elevation.
- **Step 3 — Produce:** a structured JSON package, a GeoJSON, a searchable view,
  and a summary (deterministic, plus an optional AI summary).

> **Non-operational / research only.** Nothing here is authoritative aeronautical
> data and must never be used for navigation. The VOBL chart is AAI/BIAL
> copyrighted material — use it only if you are permitted to. Holding positions
> are produced as **review-only candidates**, not accepted data.


## 0. Install

In [ ]:
%pip -q install pymupdf requests
%pip -q install 'git+https://github.com/yashpatle23/Airport-OCR.git@feat/airport-ocr-poc'

import airport_ocr
print('airport_ocr', airport_ocr.__version__, '| operational_use =', airport_ocr.OPERATIONAL_USE)

## Step 1 — Copy file

Downloads the chart and saves it as `VOBL-ADC.pdf`.

Source: https://aim-india.aai.aero/eaip/eaip-v2-01-2026/eAIP/VOBL-ADC.pdf

The AAI server sometimes blocks non-browser requests (HTTP 403). If the download
fails, the next cell lets you upload the PDF manually.

In [ ]:
import os, requests

PDF_URL = 'https://aim-india.aai.aero/eaip/eaip-v2-01-2026/eAIP/VOBL-ADC.pdf'
SRC = 'VOBL-ADC.pdf'
ok = False
try:
    r = requests.get(PDF_URL, timeout=60,
                     headers={'User-Agent': 'Mozilla/5.0', 'Accept': 'application/pdf,*/*'})
    ctype = r.headers.get('Content-Type', '')
    if r.status_code == 200 and (r.content[:5] == b'%PDF-' or 'pdf' in ctype.lower()):
        open(SRC, 'wb').write(r.content)
        ok = True
        print('downloaded', len(r.content), 'bytes ->', SRC)
    else:
        print('download not usable: HTTP', r.status_code, '| content-type', ctype)
except Exception as e:
    print('download failed:', e)

if not ok:
    print('\nFalling back to manual upload — choose your VOBL-ADC.pdf ...')
    from google.colab import files
    up = files.upload()
    picked = next((n for n in up if n.lower().endswith('.pdf')), None)
    assert picked, 'No PDF uploaded.'
    if picked != SRC:
        os.replace(picked, SRC)
    print('using uploaded file ->', SRC)

# sanity check
import pymupdf
doc = pymupdf.open(SRC)
print('pages:', doc.page_count)

### Intake — integrity & provenance (SHA-256)

In [ ]:
import json
from airport_ocr.intake import intake_file

intake = intake_file(SRC)
print(json.dumps(intake.manifest(), indent=2))
print('\nSHA-256:', intake.sha256)

## Step 2 — Extract

### 2a. Airport, runways, taxiways, coordinates/elevation (native text)

Reads the PDF word layer and reconstructs the airport header, ARP, elevation,
the runway table, and the taxiway inventory from the legend.

In [ ]:
import pymupdf, json
from airport_ocr.pdf_words import extract_from_words

doc = pymupdf.open(SRC)
pages = [{'page': i, 'size': [p.rect.width, p.rect.height], 'words': p.get_text('words')}
         for i, p in enumerate(doc)]
json.dump(pages, open('vobl_words.json', 'w'))

observations = extract_from_words(pages, dataset_id='vobl-adc-full-pipeline')
print('ICAO       :', observations['airport_icao'])
print('runways    :', [r['designator_pair'] for r in observations['runways']])
print('taxiways   :', len(observations['taxiways']['features']))
print('holding    :', observations['runway_holding_positions']['completeness_status'], '(text layer)')

### 2b. Runway holding positions — review-only candidates (vector layer)

Holding markings live in the black linework layer (not colour-separable), so
these are **candidates only** (false positives expected) and require review.

In [ ]:
from airport_ocr.holding import holding_candidates

def hexc(c):
    return None if not c else '#%02x%02x%02x' % tuple(int(round(v*255)) for v in c[:3])

page0 = doc[0]
segs = []
for d in page0.get_drawings():
    if hexc(d.get('color')) != '#000000' and hexc(d.get('fill')) != '#000000':
        continue
    for it in d['items']:
        if it[0] == 'l':
            p1, p2 = it[1], it[2]
            segs.append((p1.x, p1.y, p2.x, p2.y))
        elif it[0] == 're':
            r = it[1]
            segs += [(r.x0, r.y0, r.x1, r.y0), (r.x1, r.y0, r.x1, r.y1),
                     (r.x1, r.y1, r.x0, r.y1), (r.x0, r.y1, r.x0, r.y0)]

known = set(f['designator'] for f in observations['taxiways']['features'])
labels = []
for w in page0.get_text('words'):
    tok = w[4].strip().strip('.,&')
    if tok in known:
        labels.append({'designator': tok, 'x': (w[0] + w[2]) / 2, 'y': (w[1] + w[3]) / 2})

holding = holding_candidates(segs, labels, airport_icao=observations['airport_icao'],
                             page_size=[page0.rect.width, page0.rect.height],
                             cell=14.0, min_segments=6, max_label_distance=80.0)
print('black segments   :', holding['detector']['input_segment_count'])
print('marking-sized    :', holding['detector']['marking_segment_count'])
print('holding CANDIDATES:', holding['detector']['candidate_count'], '(NEEDS_REVIEW)')

### Identify + validate

Deterministic normalization (DMS -> CRS84), reciprocal-runway / dimension / unit
checks, elevation-conflict preservation, and taxiway-inventory validation.

In [ ]:
from airport_ocr.pipeline import normalize

normalized, geojson, report = normalize(observations)
print('VALIDATION:', report['status'], '| failures:', report['failure_count'])
print('counts    :', report['counts'])

## Step 3 — Produce (structure)

Assemble one machine-readable package covering all five groups, plus GeoJSON.

In [ ]:
from airport_ocr.report import build_package

package = build_package(normalized, report, holding_candidates=holding)
json.dump(package, open('vobl_package.json', 'w'), ensure_ascii=False, indent=2)
json.dump(geojson, open('vobl_features.geojson', 'w'), ensure_ascii=False, indent=2)

print('package groups:')
print('  airport             :', package['airport']['icao'], '-', package['airport']['name'])
print('  coordinates/elev    :', package['airport']['coordinates_elevation']['arp']['coordinates_lonlat'],
      '| elevation conflict:', package['airport']['coordinates_elevation']['elevation']['conflict_status'])
print('  runways             :', [r['designator_pair'] for r in package['runways']])
print('  taxiways            :', package['taxiways']['count'])
print('  holding (candidates):', package['runway_holding_positions']['candidate_count'], '(review-only)')

## Step 3 — Search

Query the structured GeoJSON projection by type / designator / bounding box.

In [ ]:
from airport_ocr.search import search_features

print('thresholds :', search_features(geojson, feature_type='runway_threshold')['properties']['match_count'])
print('09L        :', search_features(geojson, designator='09L')['properties']['match_count'])
bbox = [77.60, 13.10, 77.80, 13.30]
print('in bbox    :', search_features(geojson, bbox=bbox)['properties']['match_count'], 'features')

## Step 3 — Summary (deterministic)

Built directly from the structured package — nothing invented.

In [ ]:
from airport_ocr.report import summarize
from IPython.display import Markdown, display

summary_md = summarize(package)
open('vobl_summary.md', 'w').write(summary_md)
display(Markdown(summary_md))

## Step 3 — Summary + polished report (optional AI)

Uses an LLM **only** to paraphrase the already-structured package (it must not
invent or correct values, and treats chart text as untrusted). Runs only if a
`GEMINI_API_KEY` is available; otherwise it falls back to the built-in
deterministic summary.

Either way, the cell renders the whole package as one **self-contained styled
HTML card** (`render_html`) and saves it to `vobl_report.html` so you can share
a clean, non-operational report.

Add your key in Colab: **left sidebar → 🔑 Secrets → `GEMINI_API_KEY`**.


In [ ]:
# === VOBL report: Gemini paraphrase + polished HTML render ===
from airport_ocr.report import ai_summary_prompt, render_html
from IPython.display import HTML, display

ai_text = None

# 1) Get the key from Colab secrets (optional).
try:
    from google.colab import userdata
    gemini_api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    gemini_api_key = None

# 2) Ask Gemini for a paraphrase of the *already-structured* package.
if gemini_api_key:
    try:
        %pip -q install google-generativeai
        import google.generativeai as genai
        genai.configure(api_key=gemini_api_key)

        prompt = ai_summary_prompt(package)
        model = genai.GenerativeModel(
            'models/gemini-flash-latest',
            system_instruction=prompt['system'],
        )
        resp = model.generate_content(
            prompt['user'],
            generation_config={'temperature': 0},
        )
        ai_text = resp.text
        open('vobl_summary_ai.md', 'w').write(ai_text)
        print('✓ Gemini summary generated.')
    except Exception as e:
        print('AI summary skipped (using deterministic fallback):', e)
else:
    print('No GEMINI_API_KEY — using the deterministic summary (that is fine).')

# 3) Render the whole package as one styled card (AI text if we have it,
#    otherwise the built-in deterministic summary).
html = render_html(package, ai_text=ai_text)
with open('vobl_report.html', 'w') as f:
    f.write(html)
print('Saved → vobl_report.html')

display(HTML(html))


## Map (provisional)

ARP + runway thresholds. Dashed lines are threshold connectors, not surveyed
runway extents.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for f in geojson['features']:
    g = f['geometry']; p = f['properties']
    if g['type'] == 'Point':
        x, y = g['coordinates']
        is_arp = p['feature_type'] == 'aerodrome_reference_point'
        plt.scatter([x], [y], s=80 if is_arp else 35,
                    c='tab:blue' if is_arp else 'tab:green', zorder=3)
        plt.annotate('ARP' if is_arp else p.get('designator', ''), (x, y),
                     textcoords='offset points', xytext=(5, 5), fontsize=9)
    elif g['type'] == 'LineString':
        xs = [c[0] for c in g['coordinates']]; ys = [c[1] for c in g['coordinates']]
        plt.plot(xs, ys, '--', c='gray', zorder=1)
plt.xlabel('longitude'); plt.ylabel('latitude')
plt.title('VOBL ARP + runway thresholds — provisional, non-operational')
plt.grid(True, alpha=0.3); plt.gca().set_aspect('equal', adjustable='datalim'); plt.show()

## Download the outputs

In [ ]:
from google.colab import files
for name in ['vobl_package.json', 'vobl_features.geojson', 'vobl_summary.md']:
    files.download(name)

## Notes & remaining blockers

- **Non-operational**: research only; not for navigation.
- **Rights**: confirm permission for the AAI/BIAL source before storing/sharing.
- **Holding positions**: review-only candidates (black-linework clustering); expect
  false positives from dashed centrelines — human review is required.
- **Elevation conflict**: if the chart value differs from the eAIP text value, it is
  preserved (never auto-resolved).
- **AI summary**: paraphrase of the structured package only; not an authoritative source.

Source & PR: <https://github.com/yashpatle23/Airport-OCR/pull/1>
